# Bird Migration — Phase 8: Model Evaluation
**Author:** Jayant Pandey  
**Project:** Interpretable Machine Learning Application in Bird Migration Trajectory Analysis Using GPS Data  
**Group:** Group C — ISI Kolkata IDEAS Internship 2026  

---

## Evaluation Strategy

**Primary evaluation: TimeSeriesSplit Cross-Validation (5 folds)**  
As specified in the workflow, TimeSeriesSplit is used to avoid data leakage. Each fold trains on past data and tests on future data, respecting the temporal ordering of GPS records.

**Metrics computed:** Accuracy, Precision, Recall, F1 Score, ROC-AUC, Confusion Matrix (per fold and averaged)  
**Clustering metrics:** Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Score on Phase 5 K-Means zones

| Input | File |
|-------|------|
| Best model | `best_model.pkl` (DecisionTreeClassifier, max_depth=8) |
| Feature columns | `feature_cols.pkl` (40 timestep features, t-5 to t-1) |
| Sequence data | `bird_migration_sequence1.csv` (fixed — 0 duplicates) |
| Clustered data | `bird_migration_clustered__1_.csv` (fixed — 0 duplicates) |

## 0. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

N_SPLITS = 5
SEED     = 42
print('Imports done.')
print(f'TimeSeriesSplit folds: {N_SPLITS}')

Imports done.
TimeSeriesSplit folds: 5


## 1. Load Model and Feature Columns

In [2]:
best_model   = joblib.load('best_model.pkl')
feature_cols = joblib.load('feature_cols.pkl')

print('Model loaded:', type(best_model).__name__)
print(f'  max_depth : {best_model.max_depth}')
print(f'  criterion : {best_model.criterion}')
print()
print(f'Feature columns: {len(feature_cols)}')
print(f'Timesteps: t-5 to t-1 (5 past GPS observations)')
print(f'Features per timestep: 8 (latitude, longitude, altitude_clipped, speed_2d, month, hour, season, is_resting)')

Model loaded: DecisionTreeClassifier
  max_depth : 8
  criterion : gini

Feature columns: 40
Timesteps: t-5 to t-1 (5 past GPS observations)
Features per timestep: 8 (latitude, longitude, altitude_clipped, speed_2d, month, hour, season, is_resting)


## 2. Load Sequence Dataset

In [3]:
df = pd.read_csv('bird_migration_sequence.csv')
df = df.sort_values('date_time').reset_index(drop=True)
df = df.dropna(subset=['target'])

X = df[feature_cols].values
y = df['target'].astype(int).values

print('Sequence dataset loaded (fixed — 0 duplicates).')
print(f'Shape: {df.shape}')
print(f'Features: {X.shape[1]}')
print(f'Classes: {sorted(np.unique(y))}')
print()
print('Zone distribution:')
unique, counts = np.unique(y, return_counts=True)
zone_map = {0: 'Europe/Netherlands', 1: 'North Africa Transit', 2: 'West Africa Wintering'}
#zone_map = {0: 'Zone 0 (West Africa)', 1: 'Zone 1 (Netherlands)', 2: 'Zone 2 (Atlantic corridor)'}
for u, c in zip(unique, counts):
    print(f'  {zone_map[u]}: {c} records ({c/len(y)*100:.1f}%)')

Sequence dataset loaded (fixed — 0 duplicates).
Shape: (61905, 43)
Features: 40
Classes: [np.int64(0), np.int64(1), np.int64(2)]

Zone distribution:
  Europe/Netherlands: 18952 records (30.6%)
  North Africa Transit: 15407 records (24.9%)
  West Africa Wintering: 27546 records (44.5%)


## 3. TimeSeriesSplit Cross-Validation — All Metrics

Running the best model (DecisionTreeClassifier) through 5-fold TimeSeriesSplit.  
Each fold: train on past, test on future — no data leakage.

In [4]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

fold_results = []
all_cm       = np.zeros((3, 3), dtype=int)

print(f'Running {N_SPLITS}-fold TimeSeriesSplit evaluation...')
print()

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    best_model.fit(X_train, y_train)
    y_pred       = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)

    classes    = sorted(np.unique(y))
    y_test_bin = label_binarize(y_test, classes=classes)
    if len(np.unique(y_test)) == 3:
        auc = roc_auc_score(y_test_bin, y_pred_proba, multi_class='ovr', average='macro')
    else:
        auc = np.nan

    cm = confusion_matrix(y_test, y_pred, labels=[0,1,2])
    all_cm += cm

    fold_results.append({
        'Fold': fold,
        'Train size': len(train_idx),
        'Test size': len(test_idx),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc
    })

    print(f'Fold {fold}: Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f}')

results_df = pd.DataFrame(fold_results).set_index('Fold')
print()
print('Done.')

Running 5-fold TimeSeriesSplit evaluation...

Fold 1: Acc=1.0000 | Prec=1.0000 | Rec=1.0000 | F1=1.0000 | AUC=nan
Fold 2: Acc=0.9940 | Prec=0.9950 | Rec=0.9945 | F1=0.9947 | AUC=0.9952
Fold 3: Acc=1.0000 | Prec=1.0000 | Rec=1.0000 | F1=1.0000 | AUC=nan
Fold 4: Acc=1.0000 | Prec=1.0000 | Rec=1.0000 | F1=1.0000 | AUC=nan
Fold 5: Acc=0.9984 | Prec=0.9980 | Rec=0.9981 | F1=0.9980 | AUC=0.9994

Done.


## 4. Cross-Validation Results Summary

In [5]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

means = results_df[metric_cols].mean()
stds  = results_df[metric_cols].std()

print('=' * 62)
print(f'  Model: {type(best_model).__name__} | {N_SPLITS}-Fold TimeSeriesSplit')
print('=' * 62)
for col in metric_cols:
    print(f'  {col:<12}: {means[col]:.4f}  ± {stds[col]:.4f}')
print()
print('Per-fold breakdown:')
print(results_df[metric_cols].round(4).to_string())

  Model: DecisionTreeClassifier | 5-Fold TimeSeriesSplit
  Accuracy    : 0.9985  ± 0.0026
  Precision   : 0.9986  ± 0.0022
  Recall      : 0.9985  ± 0.0024
  F1 Score    : 0.9986  ± 0.0023
  ROC-AUC     : 0.9973  ± 0.0030

Per-fold breakdown:
      Accuracy  Precision  Recall  F1 Score  ROC-AUC
Fold                                                
1       1.0000      1.000  1.0000    1.0000      NaN
2       0.9940      0.995  0.9945    0.9947   0.9952
3       1.0000      1.000  1.0000    1.0000      NaN
4       1.0000      1.000  1.0000    1.0000      NaN
5       0.9984      0.998  0.9981    0.9980   0.9994


## 5. Confusion Matrix (Aggregated Across All Folds)

In [6]:
#class_labels = ['Zone 0 (W.Africa)', 'Zone 1 (Netherlands)', 'Zone 2 (Atlantic)'] changed to this
class_labels = ['Europe/Netherlands', 'North Africa Transit', 'West Africa Wintering']
fig = ff.create_annotated_heatmap(
    z=all_cm,
    x=class_labels,
    y=class_labels,
    colorscale='Blues',
    showscale=True
)
fig.update_layout(
    title=f'Confusion Matrix — Aggregated Across All {N_SPLITS} Folds | {type(best_model).__name__}',
    xaxis_title='Predicted Zone',
    yaxis_title='Actual Zone',
    height=450
)
fig.write_html('phase8_confusion_matrix.html')
fig.show()

print('Aggregated confusion matrix (all folds combined):')
print(pd.DataFrame(all_cm, index=class_labels, columns=class_labels).to_string())

Aggregated confusion matrix (all folds combined):
                       Europe/Netherlands  North Africa Transit  West Africa Wintering
Europe/Netherlands                  10664                    15                      0
North Africa Transit                    5                 13380                     59
West Africa Wintering                   0                     0                  27462


## 6. Metrics Visualisation

##### Bar chart of mean metrics

In [7]:
colors = ['#1D9E75' if means[m] >= 0.8 else '#E8542A' for m in metric_cols]

fig = go.Figure(go.Bar(
    x=metric_cols,
    y=[means[m] for m in metric_cols],
    error_y=dict(type='data', array=[stds[m] for m in metric_cols], visible=True),
    marker_color=colors,
    text=[f'{means[m]:.4f}' for m in metric_cols],
    textposition='outside'
))
fig.add_hline(y=0.8, line_dash='dash', line_color='gray',
              annotation_text='0.8 reference', annotation_position='right')
fig.update_layout(
    title=f'Phase 8 — Mean CV Metrics (±std) | {type(best_model).__name__} | {N_SPLITS}-Fold TimeSeriesSplit',
    yaxis=dict(range=[0, 1.15], title='Score'),
    height=430
)
fig.write_html('phase8_metrics_summary.html')
fig.show()

***Per-fold accuracy line chart***

In [8]:
fold_labels = [f'Fold {i}' for i in range(1, N_SPLITS+1)]

fig = go.Figure()
colors_line = ['#1D9E75', '#534AB7', '#E8542A', '#F4A261', '#065A82']

for i, metric in enumerate(['Accuracy', 'F1 Score', 'ROC-AUC']):
    fig.add_trace(go.Scatter(
        x=fold_labels,
        y=results_df[metric].values,
        mode='lines+markers',
        name=metric,
        marker=dict(size=8),
        line=dict(color=colors_line[i])
    ))

fig.update_layout(
    title=f'Key Metrics per Fold — {type(best_model).__name__} | TimeSeriesSplit',
    xaxis_title='Fold',
    yaxis=dict(range=[0, 1.05], title='Score'),
    height=420
)
fig.write_html('phase8_metrics_per_fold.html')
fig.show()

## 7. Clustering Metrics (Phase 5 K-Means Validation)

In [9]:
clustered_df = pd.read_csv('bird_migration_clustered.csv')

print('Clustered dataset loaded (fixed).')
print('Shape:', clustered_df.shape)
print('Cluster distribution:')
print(clustered_df['cluster_id'].value_counts().sort_index())

Clustered dataset loaded (fixed).
Shape: (61920, 13)
Cluster distribution:
cluster_id
0    18967
1    15407
2    27546
Name: count, dtype: int64


In [10]:
coords = clustered_df[['latitude', 'longitude']].values
labels = clustered_df['cluster_id'].values

sil_score = silhouette_score(coords, labels, sample_size=10000, random_state=SEED)
db_score  = davies_bouldin_score(coords, labels)
ch_score  = calinski_harabasz_score(coords, labels)

print('=' * 55)
print('K-Means Clustering Quality (Phase 5, k=3)')
print('=' * 55)
print(f'Silhouette Score      : {sil_score:.4f}  (higher better, max 1.0)')
print(f'Davies-Bouldin Index  : {db_score:.4f}   (lower better)')
print(f'Calinski-Harabasz     : {ch_score:.2f}  (higher better)')

K-Means Clustering Quality (Phase 5, k=3)
Silhouette Score      : 0.8296  (higher better, max 1.0)
Davies-Bouldin Index  : 0.2266   (lower better)
Calinski-Harabasz     : 1115121.14  (higher better)


In [11]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=[
        'Silhouette Score<br>(higher better)',
        'Davies-Bouldin Index<br>(lower better)',
        'Calinski-Harabasz<br>(higher better)'
    ])

fig.add_trace(go.Bar(x=['Silhouette'], y=[sil_score],
    marker_color='#1D9E75', text=[f'{sil_score:.4f}'],
    textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=['Davies-Bouldin'], y=[db_score],
    marker_color='#E8542A', text=[f'{db_score:.4f}'],
    textposition='outside'), row=1, col=2)
fig.add_trace(go.Bar(x=['Calinski-Harabasz'], y=[ch_score],
    marker_color='#534AB7', text=[f'{ch_score:.2f}'],
    textposition='outside'), row=1, col=3)

fig.update_layout(
    title='Phase 5 — K-Means Clustering Quality Metrics (k=3 migration zones)',
    height=420, showlegend=False
)
fig.write_html('phase8_clustering_metrics.html')
fig.show()

## 8. Save Evaluation Report

In [12]:
report_lines = [
    'Bird Migration — Phase 8 Model Evaluation Report',
    'ISI Kolkata IDEAS Internship 2026 — Group C',
    '=' * 60,
    f'Model          : {type(best_model).__name__}',
    f'Validation     : TimeSeriesSplit ({N_SPLITS} folds)',
    '',
    'CROSS-VALIDATION METRICS (mean ± std):',
]
for col in metric_cols:
    report_lines.append(f'  {col:<12}: {means[col]:.4f} ± {stds[col]:.4f}')

report_lines += [
    '',
    'CLUSTERING METRICS (Phase 5 K-Means, k=3):',
    f'  Silhouette Score     : {sil_score:.4f}',
    f'  Davies-Bouldin Index : {db_score:.4f}',
    f'  Calinski-Harabasz    : {ch_score:.2f}',
]

with open('evaluation_report.txt', 'w') as f:
    f.write('\n'.join(report_lines))

print('=== OUTPUT FILES ===')
output_files = [
    'evaluation_report.txt',
    'phase8_confusion_matrix.html',
    'phase8_metrics_summary.html',
    'phase8_metrics_per_fold.html',
    'phase8_clustering_metrics.html',
]
for fname in output_files:
    status = '✅' if os.path.exists(fname) else '❌ MISSING'
    print(f'  {status}  {fname}')

=== OUTPUT FILES ===
  ✅  evaluation_report.txt
  ✅  phase8_confusion_matrix.html
  ✅  phase8_metrics_summary.html
  ✅  phase8_metrics_per_fold.html
  ✅  phase8_clustering_metrics.html
